# មនុស្សក្នុងខ្សែបញ្ជា៖ ក្រែងសកម្មភាពមុន, ការតម្រៀបហានិភ័យ, និងកំណត់ហេតុត្រួតពិនិត្យ

README សម្រាប់មេរៀននេះណែនាំមនុស្សក្នុងខ្សែបញ្ជា ជាមួយខ្លឹមសារសង្ខេបខ្លី ដែលស្នើឱ្យអ្នកប្រើ `APPROVE` ឬ `REJECT` បន្ទាប់ពីភ្នាក់ងារបានបង្កើតចម្លើយរួច។ តម្រង់ទិសនោះគឺជាចំណុចចាប់ផ្តើមល្អ ប៉ុន្តែការអនុវត្ត HITL ក្នុងផលិតកម្មជាច្រើនត្រូវការផ្នែកបន្ថែមបីយ៉ាង៖

1. **ក្រឡាដែនសកម្មភាពមុន** ដែលដំណើរការមុន **មនុស្សភ្នាក់ងារធ្វើជំហានហានិភ័យ ដូចនេះថ្លៃដើម, មិនអាចត្រឡប់ក្រោយបាន, និងយឺតយ៉ាវ ទទួលបានការត្រួតត្រា។**
2. **ការតម្រៀបថ្នាក់ហានិភ័យ**, ដូច្នេះសកម្មភាពហានិភ័យទាបអនុវត្តដោយស្វ័យប្រវត្តិ សកម្មភាពហានិភ័យមធ្យមត្រូវបានអនុម័តជាក្រុម និងសកម្មភាពហានិភ័យខ្ពស់ត្រូវរាំងស្ទះដោយមនុស្ស។
3. កំណត់ហេតុត្រួតពិនិត្យ និងរង្វិលបញ្ចូលការ បែប សម្រាប់ការសម្រេចចិត្តគ្រប់ក្រឡាដែនត្រូវបានកត់ត្រាជា JSONL រួចការបដិសេធនឹងធ្វើឱ្យភ្នាក់ងារត្រូវបញ្ចូលឡើងវិញជាមួយហេតុផលរចនាសម្ព័ន្ធជំនួសមានតែ `Revising...`។

សៀវភៅកំណត់នេះបង្កើតនូវផ្សំនេះទាំងអស់ជាសំណុំដូចជា `06-system-message-framework.ipynb`។ វាដំណើរការពីដើមដល់ចុង ក្នុង `DEMO_MODE = True` (មិនត្រូវការបញ្ចូលផ្ទាល់) ឬជាមួយនឹងការស្នើរសុំ `input()` ពិតប្រាកដនៅពេល `DEMO_MODE = False`។ ទំរង់សំខាន់៖ នៅក្នុង DEMO_MODE ការបញ្ចុះបញ្ចូលលើគោលបំណងទីបី ត្រូវបានសរសេរជាកូដ ដូច្នេះគីណេម៉ាទិចរង្វិលឃើញបានចុងពីដើមដល់ចុង។ ការប្រើប្រាស់វិធីសាស្រ្តបញ្ជូនពិន្ទុវិញជាក់ស្តែងត្រូវការឲ្យ DEMO_MODE = False និងប្រតិបត្តិករសកម្មភាព។

**ចេញពីវិស័យ (ដោះស្រាយក្នុងមេរៀនផ្សេងទៀត):** ការផ្ទៀងផ្ទាត់ និងការគ្រប់គ្រងការចូល (មេរៀន 06 README ឆ្លងកាត់គ្រោះថ្នាក់ 2), កម្មវិធីកណ្តាលគ្រប់គ្រងការហៅឧបករណ៍ (មេរៀន 14 MAF ជ្រៅជម្រើស), លំនាំវេទិកាគំនិតធ្វើការ​ដោយភ្នាក់ងារច្រើន។


In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

# This notebook uses the Azure OpenAI Responses API via the stable /openai/v1/ endpoint.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API.
endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
if not endpoint:
    raise RuntimeError(
        "AZURE_OPENAI_ENDPOINT environment variable is not set. This notebook needs "
        "an Azure OpenAI resource with a model deployment that supports the Responses "
        "API. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_DEPLOYMENT in "
        "your environment or a local .env file, then run `az login`."
    )

deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# Authenticate with Entra ID (run `az login` first). No api_version is needed.
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

client = OpenAI(
    base_url=f"{endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)



## លំនាំទី 1: ទ្វារមុនសកម្មភាព

ឧទាហរណ៍ HITL ក្នុង README បានហៅតំណាងជាមុន រួចបានស្នើឱ្យអ្នកប្រើអនុម័តលទ្ធផល។ នេះគឺជាទ្រង់ទ្រាយ **បន្ទាប់សកម្មភាព**។ តំណាងបានអនុវត្តរួចហើយ ដូច្នេះចំណាយក្នុងការហៅ LLM ត្រូវបានបង់រួច ហើយប៉ះពាល់ណាមួយ (ផ្ញើអ៊ីមែល បានសរសេរជួរទិន្នន័យ រឺបានផ្សាយមតិយោបល់) បានកើតឡើងរួចហើយ។

ទ្រង់ទ្រាយ **មុនសកម្មភាព** បញ្ចូលទ្វារមុនពេលតំណាងដំណើរការជំហានមានហានិភ័យ។ តំណាងស្នើសុំសកម្មភាព ទ្វារសំរេចថាតើត្រូវអនុវត្ត ឬទេ ហើយមានតែពេលអនុម័តប៉ុណ្ណោះដែលប៉ះពាល់នឹងកើតមាន។

| ផ្នែក | ការអនុម័តបន្ទាប់សកម្មភាព (ឧទាហរណ៍ README) | ទ្វារមុនសកម្មភាព (សៀវភៅកំណត់ត្រានេះ) |
|---|---|---|
| អានុម័តរត់នៅពេលណា? | បន្ទាប់ពីតំណាងបានបង្កើតលទ្ធផល | មុនសកម្មភាពណាមួយកើតឡើង |
| ចំណាយ LLM នៅពេលបដិសេធ | បានបង់រួចហើយ | បង់ត្រឹមសម្រាប់ការស្នើសុំ ប៉ុន្មានសកម្មភាព |
| ផលប៉ះពាល់មិនអាចបង្វិល | អាចកើតឡើង (សកម្មភាពបានកើតរួចហើយ) | បានការពារ |
| ភាពច្បាស់លាស់សម្រាប់ការត្រួតពិនិត្យ | ការអនុម័តគឺជាអត្ថបទបោះពុម្ភ | ការអនុម័តគឺជាកំណត់ត្រា JSON មានពេលវេលា សកម្មភាព និងហេតុផល |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## គំរូទី 2៖ ការវាយតម្លៃហានិភ័យតាមជាន់

មិនមែនសកម្មភាពគ្រប់យ៉ាងត្រូវការឲ្យមនុស្សអនុម័តទេ។ ការស្វែងរកតាមអានតែប៉ុណ្ណោះប្រឆាំងនឹង API សាធារណៈមានភាពខុសគ្នាក្នុងផលប៉ះពាល់ជាមួយការផ្ញើអ៊ីមែលទៅអតិថិជន។ ការប្រព្រឹត្តទៅដូចគ្នាជាទៀងទាត់ធ្វើឲ្យផ្តោតអារម្មណ៍របស់អ្នកប្រតិបត្តិការលំបាកហើយធ្វើឲ្យភ្នាក់ងារអាប់ដោន។

គំរូ៣ជាន់សាមញ្ញ៖

| ជាន់ | ឧទាហរណ៍ | សេចក្តីដំណឹងអនុម័ត |
|---|---|---|
| `ទាប` (អានតែប៉ុណ្ណោះ) | ស្វែងរកមូលដ្ឋានចំណេះដឹង, ស្វែងរកជម្រើសអ៊ែរលាញ់, ទាញយកទំព័របណ្ដាញសាធារណៈ | ប្រតិបត្តដោយស្វ័យប្រវត្តិ, បានកត់ត្រាសម្រាប់ត្រួតពិនិត្យ |
| `មធ្យម` (ផ្លាស់ប្ដូរថ្លៃថោក) | ការកត់ត្រាលទ្ធផល, ការកើនឡើងឧបករណ៍រាប់, ការកំណត់កាលបរិច្ឆេទរំលឹក | ប្រតិបត្តដោយស្វ័យប្រវត្តិ, ប៉ុន្តែមានការត្រួតពិនិត្យជាប្រចាំថ្ងៃ |
| `ខ្ពស់` (ជំនួញរួមផ្សេង ឬមិនអាចត្រឡប់ក្រោយ) | ផ្ញើអ៊ីមែល, ចោទថ្លៃកាត, បង្ហោះទៅបណ្ដាញផ្សព្វផ្សាយសាធារណៈ | ហាមឃាត់រង់ចាំការអនុម័តពីមនុស្ស |

នេះជាជាន់មួយ។ ប្រព័ន្ធផលិតកម្មភាគច្រើនប្រើជាន់លំអិតជាងនេះ (ឧទាហរណ៍: កម្រិតការអនុញ្ញាត AWS IAM, ជាន់សុវត្ថិភាពមូលដ្ឋានតួនាទី)។ គំរូ ៣ជាន់ខាងក្រោមគឺជាគំរូតូចបំផុតដែលមានប្រយោជន៍សម្រាប់ភ្នាក់ងារដែលច្របូកច្របល់សកម្មភាពអានតែប៉ុណ្ណោះ និងនិងមុខងារស្របតាមប្រតិបត្តិការ។

ម៉ាស៊ីនចាត់ថ្នាក់ខាងក្រោមប្រើវិធីសាស្ត្រកំណត់ពាក្យនិយមដើម្បីឲ្យកម្មវិធីប្រើប្រាស់បានមានលទ្ធផលច្បាស់លាស់ និងថ្លៃថោក។ ក្នុងប្រព័ន្ធផលិតកម្មអ្នកអាចជំនួសវាជាមួយម៉ាស៊ីនចាត់ថ្នាក់រៀនឬម៉ាស៊ីនគ្រប់គ្រងគោលនយោបាយ។


In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


##រូបមន្តទី 3៖ កំណត់ហេតុសវនាការនិងឧបករណ៍ពិនិត្យលើកម្ដងម្កាល

`print("Response approved.")` មិនមែនជាកំណត់ហេតុសវនាការទេ។ សម្រាប់ការជឿជាក់ សេចក្ដីសម្រេចចិត្តនៅគ្រប់ច្រកគួរត្រូវបានរក្សាទុកជាផ្នែកមួយនៃព្រឹត្តិការណ៍ដែលមានរូបមន្ត ដែលអ្នកអាចស្វែងរក​វិញ បានច្រៀង​ឡើងវិញ ឬភ្ជាប់ទៅនឹងការត្រួតពិនិត្យករណីបន្ទាប់។

ចំនួនពីរផ្នែក៖

1. **JSONL បន្ថែមតែមួយទំព័រ។** មួយជួរដួងសម្រាប់សេចក្ដីសម្រេចមួយ ដោយមានពេលវេលា សកម្មភាព ស្រទាប់ សេចក្ដីសម្រេច និងមូលហេតុ។ ងាយស្រួលស្វែងរក និងងាយស្រួលផ្ញើទៅឃ្លាំងកំណត់ហេតុពិតប្រាកដបន្ទាប់។
2. **ឧបករណ៍ពិនិត្យឡើងវិញនៅពេលបដិសេធ។** ពេលដែលច្រកបញ្ជូនតម្លៃ `deny` មកវិញ អ្នកភ្នាក់ងារនាំខ្លួនឯងមកស្នើរសុំម្ដងទៀតជាមួយមូលហេតុបដិសេធនៅក្នុងបរិបទ ដើម្បីឱ្យការស្នើរបន្ទាប់បន្ទាប់អាចជៀសវាងបញ្ហា។


In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.responses.create(
        model=deployment,
        input=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_text},
        ],
        store=False,
    )
    return response.output_text.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}



In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## ឧបករណ៍បន្ថែម

គម្រោងសាធារណៈជាច្រើនផ្សេងទៀតអនុវត្តបំលែងនៃគំរូ HITL ទាំងនេះ។ ប្រៀបធៀបវិធីសាស្ត្រដើម្បីរកអ្វីដែលសមរម្យសម្រាប់គេហទំព័ររបស់អ្នកៈ

- **LangChain** ការរុំតួឧបករណ៍ human-in-the-loop ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)): ការរុំតួឧបករណ៍ដែលផ្អាកការប្រតិបត្តិការសម្រាប់ការបញ្ចូលពីមនុស្ស។
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ បានបង្ហាញឡើងវិញ): ប្រើតួតំណាងជាអ្នកភាគីដើម្បីតំណាងឱ្យមនុស្សក្នុងការពិភាក្សាច្រើនភាគី។
- **Microsoft Agent Framework (MAF)** ការបណ្តុះបណ្តាលមុខងារសម្រាប់ការហៅ ([docs](https://learn.microsoft.com/agent-framework/)): មុខងារជាកម្មវិធីមធ្យមដែលដំណើរការជុំវិញការហៅឧបករណ៍/មុខងារណាមួយសម្រាប់ការត្រួតពិនិត្យកLogicនិងលំហាត់អនុម័ត។

គម្រោងនីមួយៗដោះស្រាយបំលែងតូចបីនេះយ៉ាងខុសគ្នា៖ LangChain រុំវាជាឧបករណ៍, AutoGen ប្រើតួតំណាង ហើយ Microsoft Agent Framework ប្រើមុខងារសម្រាប់ការហៅ។ អានការអនុវត្តន៍មួយឬពីរពីដើមដើម្បីជ្រើសរើសរចនាសម្ព័ន្ធសម្រាប់អ្នកភាគីរបស់អ្នក។


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
